# Acoustic Keystroke Robustness

Clean, Gaussian and FGSM recognition at matched SNR using one CoAtNet checkpoint.
The main test uses 30/20/10 dB; the wider validation sweep is exploratory.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/NPM26103/Research-Methodology.git"
GIT_REF = ""  # Commit or tag

ROOT = Path.cwd()
if not (ROOT / "rsm/pipeline.py").is_file():
    ROOT = Path("/content/RsM") if Path("/content").is_dir() else ROOT / "RsM"
    if not (ROOT / "rsm/pipeline.py").is_file():
        if not REPO_URL:
            raise ValueError("Set REPO_URL to this project's GitHub repository URL.")
        subprocess.run(["git", "clone", "--", REPO_URL, str(ROOT)], check=True)
        if GIT_REF:
            subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", GIT_REF], check=True)

if not (ROOT / "rsm/pipeline.py").is_file():
    raise FileNotFoundError("The repository root must contain rsm/ and RsM.ipynb.")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [ ]:
from rsm.notebook import Experiment

RUN_NAME = "seed26"
DEVICE = "cuda"
USE_DRIVE = True

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB and USE_DRIVE:
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/RsM")
else:
    BASE = ROOT

RUN_DIR = BASE / "runs" / RUN_NAME
DATA_ROOT = (Path("/content/rsm_data") if IN_COLAB else ROOT / "data") / RUN_NAME
experiment = Experiment(RUN_DIR, DATA_ROOT, device=DEVICE)
experiment.setup()
print("Run:", RUN_DIR)

## Data preprocessing

Recordings are downloaded from the pinned dataset revision and checked by SHA-256.
Audio review covers one clip per class and every `j` candidate before the split is fixed.


In [ ]:
from IPython.display import Image, display

experiment.run("prepare")
manifest = experiment.table("manifest.csv")
display(manifest.groupby("key").size().rename("clips").to_frame().T)
display(Image(filename=str(experiment.processed / "qc/j.png"), width=1000))


In [ ]:
KEY = "all"
EXTRA_REVIEW_IDS = []
experiment.review_audio(key=KEY, extra_ids=EXTRA_REVIEW_IDS)

In [ ]:
# Change the values ​​below after review
REVIEWER = ""
QC_NOTES = ""
REVIEW_COMPLETE = False
REJECTED = {}  # sample_id: reason
ALLOW_DATA_REVISION = False

experiment.freeze(
    reviewer=REVIEWER,
    notes=QC_NOTES,
    reviewed=REVIEW_COMPLETE,
    rejected=REJECTED,
    extra_ids=EXTRA_REVIEW_IDS,
    allow_revision=ALLOW_DATA_REVISION,
)
display(experiment.table("split_counts.csv"))


## Training

CoAtNet-0, Adam, batch size 16, learning rate 0.0005.
Training runs to 1,100 epochs or early stopping (patience 100); selection uses validation macro-F1.


In [ ]:
experiment.run("train")
experiment.show_training()

## Main test

Validation sanity checks verify the saved baseline and waveform gradients.
The fixed checkpoint is then evaluated on test at 30/20/10 dB, with 20 dB as the primary comparison.


In [ ]:
experiment.run("sanity")
checks = experiment.result_dir("validation_sanity") / "validation_checks.json"
display(json.loads(checks.read_text()))

In [ ]:
experiment.run("main")
main_results = experiment.show_results(
    "main_test",
    tables=["metrics_summary.csv", "paired_effects.csv"],
    figures=["accuracy_vs_snr.png", "confusion_clean.png", "confusion_fgsm_20.png"],
)


In [ ]:
MAIN_BUNDLE = experiment.export("main")


## Exploratory validation

The same checkpoint is evaluated at 60/50/40/35/30/20/10 dB.
Validation was used for checkpoint selection, so these diagnostics are reported separately from test.


In [ ]:
experiment.run("diagnostics")
validation_results = experiment.show_results(
    "exploratory_validation",
    tables=[
        "metrics_summary.csv",
        "paired_effects.csv",
        "prediction_concentration.csv",
        "frontend_shift.csv",
    ],
    figures=["accuracy_vs_snr.png", "confusion_fgsm_20.png"],
)

main_results = experiment.result_dir("main_test")
main_meta = json.loads((main_results / "evaluation_metadata.json").read_text())
val_meta = json.loads((validation_results / "evaluation_metadata.json").read_text())
assert main_meta["checkpoint_sha256"] == val_meta["checkpoint_sha256"]


In [ ]:
EXAMPLE_INDEX = 0
experiment.show_example(EXAMPLE_INDEX)

## Export

The combined bundle contains predictions, tables, figures and run metadata.
The selected model and resumable training state remain in the run directory.


In [ ]:
INCLUDE_CHECKPOINT = True
FINAL_BUNDLE = experiment.export("all", include_checkpoint=INCLUDE_CHECKPOINT)
print("Best checkpoint:", RUN_DIR / "training/best.pt")
print("Resume checkpoint:", RUN_DIR / "training/last.pt")
